# Lesson 2 - Assignment

### In this assignment, you will implement a Decision Tree algorithm from scratch and compare the results to existing sklearn algorithm. 

In [1]:
# import packages
%matplotlib inline
import numbuild_tree np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from matplotlib.legend_handler import HandlerLine2D
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# make this notebook's output stable across runs
np.random.seed(0)

In [2]:
data = pd.read_csv("data_banknote_authentication.csv", header=None)

In [3]:
data.columns

Int64Index([0, 1, 2, 3, 4], dtype='int64')

## Question 1.1: Implement the functions to calculate Gini Index.

In [4]:
groups  = [data[feature] for feature in [0, 1, 2, 3]]
classes = data[4]

In [22]:
# Calculate the Gini index for a split dataset
def gini_index(groups, classes):
    # DONE count all samples at split point
    count = sum([len(group) for group in groups])
    # DONE I THINK sum weighted Gini index for each group
    gini = 0
    for group in groups:
        size = len(group)
        
        if size == 0:# TODO avoid divide by zero
            continue
        score = 0
        # TODO score the group based on the score for each class
        
        for class_val in classes:
            p = [row for row in group].count(class_val)/size #to do calculate p
            score += p**2    
        # TODO weight the group score by its relative size
        gini += (1 - score) / (len(group)/count)
    return gini

## Question 2.1: Write a method that splits the 

In [7]:
data.mean()

0    0.433735
1    1.922353
2    1.397627
3   -1.191657
4    0.444606
dtype: float64

In [8]:
data[data[4]==0].mean()

0    2.276686
1    4.256627
2    0.796718
3   -1.147640
4    0.000000
dtype: float64

In [9]:
data[data[4]==1].mean()

0   -1.868443
1   -0.993576
2    2.148271
3   -1.246641
4    1.000000
dtype: float64

In [66]:
# Split a dataset based on an attribute and an attribute value
def test_split(index, value, dataset):
    left, right = list(), list()
    """
    TODO: This function loops over each row and checks if the row belongs to the right or left list.
    """ 
    if index in dataset.columns:
        data = dataset[index]
    else:
        data = dataset
    for row in data:
        if row < float(value):
            left.append(row)
        else:
            right.append(row)
    return left, right

## Question 2.2: Write a method that loops over the dataset, determine the groups of rows that belong to the right or left split, and calculates the gini_index

In [11]:
def get_split_lengths(dataset):
    class_values = [dataset[index].mean() for index in dataset.columns[:-1]]
    classes      = dataset.columns[:-1]
    lengths      = list()
    for index in classes:
        lengths.append([len(data) for data in test_split(index,class_values[index],dataset)])
    return lengths

In [12]:
class_lengths = get_split_lengths(data)
class_lengths

[[680, 692], [655, 717], [819, 553], [523, 849]]

In [13]:
dataset = data.drop(columns=4).values
targets = data[4].values

In [14]:
dataset[:10]

array([[ 3.6216 ,  8.6661 , -2.8073 , -0.44699],
       [ 4.5459 ,  8.1674 , -2.4586 , -1.4621 ],
       [ 3.866  , -2.6383 ,  1.9242 ,  0.10645],
       [ 3.4566 ,  9.5228 , -4.0112 , -3.5944 ],
       [ 0.32924, -4.4552 ,  4.5718 , -0.9888 ],
       [ 4.3684 ,  9.6718 , -3.9606 , -3.1625 ],
       [ 3.5912 ,  3.0129 ,  0.72888,  0.56421],
       [ 2.0922 , -6.81   ,  8.4636 , -0.60216],
       [ 3.2032 ,  5.7588 , -0.75345, -0.61251],
       [ 1.5356 ,  9.1772 , -2.2718 , -0.73535]])

In [173]:
def get_split(dataset):
    "TODO Select the best split point for a dataset"
    class_values  = [row[-1] for row in dataset]
    b_index, b_value, b_score, b_groups = 999, 999, 999, None
    for index in range(1):# OLD RANGE ARGUMENT len(dataset.columns)-1
        length = class_lengths[index]
        for row in dataset:
            groups = test_split(index,row[index],data)
            #probs  = [sum(targets.iloc[group]/len(group)) for group in groups]
            gini = gini_index(groups,class_values)    #1 - sum([prob**2 for prob in probs]) #OLD
            if gini < b_score:                                                        #I guess this makes sense assuming the below is correct, it just 'updates' when a best 'fit' is found
                b_index, b_value, b_score, b_groups = index, row[index], gini, groups #Assuming this is what 'they' want
    return {'index':b_index, 'value':b_value, 'groups':b_groups}

In [23]:
get_split(data.values)

{'index': 0,
 'value': -7.0421,
 'groups': ([],
  [3.6216,
   4.5459,
   3.866,
   3.4566,
   0.32924,
   4.3684,
   3.5912,
   2.0922,
   3.2032,
   1.5356,
   1.2247,
   3.9899,
   1.8993,
   -1.5768,
   3.404,
   4.6765,
   2.6719,
   0.80355,
   1.4479,
   5.2423,
   5.7867,
   0.3292,
   3.9362,
   0.93584,
   4.4338,
   0.7057,
   1.1432,
   -0.38214,
   6.5633,
   4.8906,
   -0.24811,
   1.4884,
   4.2969,
   -0.96511,
   -1.6162,
   2.4391,
   2.6881,
   3.6289,
   4.5679,
   3.4805,
   4.1711,
   -0.2062,
   -0.0068919,
   0.96441,
   2.8561,
   -0.7869,
   2.0843,
   -0.7869,
   3.9102,
   1.6349,
   4.3239,
   5.262,
   3.1452,
   2.549,
   4.9264,
   4.8265,
   2.5635,
   5.807,
   3.1377,
   -0.78289,
   2.888,
   0.49665,
   4.2586,
   1.7939,
   5.4021,
   2.5367,
   4.6054,
   2.4235,
   1.0009,
   0.12326,
   3.9529,
   4.1373,
   4.7181,
   4.1654,
   4.4069,
   2.3066,
   3.7935,
   0.049175,
   0.24835,
   1.1317,
   2.8033,
   4.4682,
   5.0185,
   1.8664,
   3.245

## Question 2.3: Repeat question 2.2 using entropy

In [37]:
#Not working
def get_entropy(groups,class_values):
    e = 0
    for group in groups:
        size = len(group)
        if size == 0:
            continue
        for class_value in classes:
            p = [row for row in group].count(class_value)/size #reusing from gini
            value = -p*np.log2(p)
        e += value
    return e

In [38]:
#Not Working
def get_split_entropy(dataset):
    "TODO Select the best split point for a dataset"
    class_values = [row[-1] for row in dataset]
    b_index, b_value, b_score, b_groups = 999, 999, 999, None
    for index in range(3):            #range(len(dataset[0])-1):
        for row in dataset:
            groups = test_split(index,row[index],data)
            entropy = get_entropy(groups,class_values) #FINISH ME
            if entropy < b_score:
                b_index, b_value, b_score, b_groups = index, row[index], gini, groups
    return {'index':b_index, 'value':b_value, 'groups':b_groups}

In [39]:
get_split_entropy(data.values)

/tmp/ipykernel_15967/2434385275.py:10: RuntimeWarning: divide by zero encountered in log2
  value = -p*np.log2(p)
/tmp/ipykernel_15967/2434385275.py:10: RuntimeWarning: invalid value encountered in double_scalars
  value = -p*np.log2(p)


{'index': 999, 'value': 999, 'groups': None}

## Question 3.1: Write a method that takes in a group of rows and determines the class they belongs to. It should return the most common output value for a list of rows.

In [46]:
def to_terminal(group):
    "TODO determing the most commong output within each group"
    outcomes = [row for row in group]
    return max(set(outcomes), key=outcomes.count)

## Question 3.2: Write a method that recursively split the data.
The method takes in a node, max_depth, min_size, and depth. Initially, the method would be called by passing the rood node and depth of 1. Here are the steps to help you implement:

- First, we create two groups for the data split and delete any existing groups from the node. As rows are used, they are no longer needed.
- Second, check if rows should be in left or right group, and if so we create a terminal node using the records we have.
- Third, check if maximum depth is reached and if so we create a terminal node.
- Fourth, process left child, creating a terminal node if the group of rows is too small, otherwise creating and adding the left node in a depth first fashion until the bottom of the tree is reached on this branch.
- Fifth, process the right side in a similar manner as left side, as we rise back up the constructed tree to the root.

In [167]:
lt = targets[data[data[0]<v].index] #left target targets
rt = targets[data[data[0]>v].index] #right target targets
#this appears to be neccesary since the left and right splits require the targets when the get_split function is applied, I'm running into issues here and I may have to give up since the deadline is fast approaching

In [174]:
# Create child splits for a node or make terminal
def split(node, max_depth, min_size, depth):
    left, right = node["groups"]
    left, right = [[left[i],lt[i]] for i in range(len(left)-1)],[[right[i],rt[i]] for i in range(len(right)-1)]
    del(node['groups'])
    # TODO check for a no split
    if not left or not right:
        node["left"] = node["right"] = to_terminal(left+right)
        return
    # TODO check for max depth
    if depth >= max_depth:
        #TODO
        node["left"], node["right"] = to_terminal(left), to_terminal(right)
        return
    # TODO process left child
    if len(left) <= min_size:
        #TODO
        node["left"] = to_terminal(left)
    else:
        node['left'] = get_split(left) #left here is filled w just the feature values and not targets, but get_split eventually requires the target values. working on this.
        split(node["left"], max_depth, min_size, depth+1)
        #TODO
    # TODO process right child
    if len(right) <= min_size:
        node["right"] = to_terminal(right)
        #TODO
    else:
        node["right"] = get_split(right)
        split(node["right"],max_depth,min_size,depth+1)

## Question 3.3: Write a method that builds the tree. The method creates an initial split to determine root node, and then calls the split method.

In [101]:
node = get_split(train)

In [79]:
left, right = get_split(train)["groups"]

In [152]:
v = float(node["value"])

In [168]:
len(left),len(lt)

(686, 686)

In [170]:
[[left[i],lt[i]] for i in range(len(left))]

[[0.32924, 0],
 [-1.5768, 0],
 [0.3292, 0],
 [-0.38214, 0],
 [-0.24811, 0],
 [-0.96511, 0],
 [-1.6162, 0],
 [-0.2062, 0],
 [-0.0068919, 0],
 [-0.7869, 0],
 [-0.7869, 0],
 [-0.78289, 0],
 [0.12326, 0],
 [0.049175, 0],
 [0.24835, 0],
 [-1.1313, 0],
 [-0.64472, 0],
 [-2.7419, 0],
 [-1.8584, 0],
 [-1.4572, 0],
 [-1.5075, 0],
 [-0.91718, 0],
 [-2.343, 0],
 [0.4339, 0],
 [-1.0401, 0],
 [-0.2062, 0],
 [-1.7599, 0],
 [0.17346, 0],
 [-1.803, 0],
 [0.11739, 0],
 [-1.6952, 0],
 [-1.1193, 0],
 [0.19081, 0],
 [-0.13144, 0],
 [-0.11783, 0],
 [-0.69572, 0],
 [-1.7559, 0],
 [-1.2537, 0],
 [-2.341, 0],
 [-1.8584, 0],
 [-2.0897, 0],
 [-0.78689, 0],
 [-0.16735, 0],
 [-1.3, 0],
 [-2.2261, 0],
 [-0.36038, 0],
 [-2.6479, 0],
 [-1.3389, 0],
 [-2.3361, 0],
 [0.46901, 0],
 [-1.3274, 0],
 [-1.3931, 0],
 [0.3798, 0],
 [-0.016103, 0],
 [0.20977, 0],
 [0.3223, 0],
 [-1.3, 0],
 [0.44125, 0],
 [-2.2153, 0],
 [0.051979, 0],
 [0.3292, 0],
 [-1.9177, 0],
 [0.27451, 0],
 [0.3292, 0],
 [-1.7344, 0],
 [-0.16108, 0],
 [-1.

In [42]:
# Build a decision tree
def build_tree(train, max_depth, min_size):
    "TODO get the first split, and then split starting fromt the root"
    root = get_split(train)
    split(root,max_depth,min_size,1)
    return root

In [47]:
tree = build_tree(data.values,1,1)
tree

{'index': 0, 'value': -7.0421, 'left': 0.5706, 'right': 0.5706}

## Question 3.4: Write a method that takes in a node and rows of data, and predicts the class associated with each row.

In [48]:
# Make a prediction with a decision tree
def predict(node, row):
    #TODO check if a row belongs to a node and recursively traverse the tree if the row doesn't.
    if row[node['index']] < node['value']:
        if isinstance(node["left"],dict):
            return predict(node["left"],row)
        else:
            return node['left']
    else:
        if isinstance(node['right'], dict):
            return predict(node["right"],row)
        else:
            return node['right']

## Question 4: Train a decision tree using the banknote_authentication data

In [49]:
from random import seed
from random import randrange
from csv import reader
from sklearn.metrics import accuracy_score

# Load a CSV file
def load_csv(filename):
    file = open(filename, "rt")
    lines = reader(file)
    dataset = list(lines)
    return dataset
 
# Convert string column to float
def str_column_to_float(dataset, column):
    for row in dataset:
        row[column] = row[column].strip()#.astype(float)
        
filename = 'data_banknote_authentication.csv'
dataset = load_csv(filename)
# convert string attributes to integers
for i in range(len(dataset[0])):
    str_column_to_float(dataset, i)
train = dataset[1:np.int(len(dataset)*2/3)]
test = dataset[np.int(len(dataset)*2/3)+1:len(dataset)]

/tmp/ipykernel_15967/3341140677.py:23: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  train = dataset[1:np.int(len(dataset)*2/3)]
/tmp/ipykernel_15967/3341140677.py:24: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in 

In [175]:
#TODO Build a tree and evalute accuracy
tree = build_tree(train, 2, 1)
predictions = list()
for row in test:
    prediction = (tree, row)
    predictions.append(prediction)               
print('Accuracy: %s' % accuracy_score([row[-1] for row in test], predictions))

IndexError: index 685 is out of bounds for axis 0 with size 685

## Question 6. Create a new text cell in your Notebook: Complete a 50-100 word summary (or short description of your thinking in applying this week's learning to the solution) of your experience in this assignment. Include:
                                                                      
* What was your incoming experience with this model, if any? 
* What steps you took, what obstacles you encountered.
* How you link this exercise to real-world, machine learning problem-solving. (What steps were missing? What else do you need to learn?) 
> This summary allows your instructor to know how you are doing and allot points for your effort in thinking and planning, and making connections to real-world work.

> I had no experience coming into this lesson with decision trees. Outside of the resources providided in class, I researched the internet extensively on decision trees, and examples of them being used in an ML context that are similar to what we're doing here. After finding similar examples and seeing how to replicate the process using the starter code provided, I was able to fill out most of the already started functions that enable to use of decision trees with the bankink data. I plan on reviewing my work here after I submit to better improve it because I'm suspicious of my results above, I'm skeptical if they're correct. My academic experience is primarily in the context of physics and astronomy so I'm having trouble envisaging an application of decision trees in those fields, I could potentially see them being applied to particle decays because there are many different decay routes a particle could take and they're all inherently probabilistic, so I could see how this concept of decision trees could be relevant to modeling a new particles resonances for instance.